## 08-06-2026 PANDAS DATA MANIPULATION
Pandas is the main tool for working with tabular data in Python. The two core objects are the Series (a single labelled column) and the DataFrame (a 2D table of Series sharing one index). Most data work is: select the right rows and columns, clean missing values, then group, combine and transform. This notebook walks through each of those.

In [1]:
import numpy as np
import pandas as pd
np.random.seed(101)

### Sample Data
The class loaded a local titanic CSV. To keep this notebook runnable anywhere I build an equivalent sample with the same columns and types. One detail: newer pandas stores text as a `str` dtype, so I cast the text columns back to `object` so the classic `dtypes == "object"` trick still works the way it did in class.

In [2]:
n = 150
first = ['Theresa','Jose','Laura','Phillip','Tina','Glenn','Kim','Morgan','Cody','Tammy',
         'Kari','Sylvia','Jimmy','Stephanie','Jennifer','Joseph','Michael','Gregory','Anna','Brian']
last = ['Davis','Clark','Mccarthy','Merritt','Mercado','Murray','Schmidt','Brown','Cox','Marsh',
        'Johnson','Skinner','Hernandez','Scott','Ortega','Wilson','Collins','Ellis','Bell','Adams']
names, seen = [], set()
while len(names) < n:
    nm = np.random.choice(first) + ' ' + np.random.choice(last)
    if nm not in seen:
        seen.add(nm); names.append(nm)

titanic_train = pd.DataFrame({
    'PassengerId': np.arange(1, n+1),
    'Survived': np.random.randint(0, 2, n),
    'Pclass': np.random.choice([1, 2, 3], n),
    'Name': names,
    'Sex': np.random.choice(['male', 'female'], n),
    'Age': np.round(np.random.uniform(1, 60, n), 1),
    'SibSp': np.random.randint(0, 6, n),
    'Parch': np.random.randint(0, 6, n),
    'Fare': np.round(np.random.uniform(10, 100, n), 2),
    'Cabin': np.random.choice(['B20','E46','G6','F2','D33','A10','C123', np.nan], n),
    'Embarked': np.random.choice(['Q', 'S', 'C'], n),
})
for col in ['Name','Sex','Cabin','Embarked']:
    titanic_train[col] = titanic_train[col].astype(object)
titanic_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,1,1,1,Sylvia Ellis,female,21.8,2,3,22.79,F2,C
1,2,0,3,Kim Skinner,male,36.8,4,4,58.05,G6,Q
2,3,1,1,Joseph Marsh,female,50.8,4,5,14.57,nan,C
3,4,1,1,Stephanie Cox,male,41.7,5,3,90.34,D33,C
4,5,0,1,Tina Cox,female,3.1,4,2,77.31,C123,Q


### Inspecting Data
The first thing with any dataset is to look at it. `head(n)` and `tail(n)` show the first and last rows. `dtypes` lists the type of each column: int64 and float64 are numbers, object is text. Knowing the types matters because numeric and text columns support different operations.

In [3]:
# dtypes - the type of each column
titanic_train.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Cabin           object
Embarked        object
dtype: object

A handy trick to grab only the text columns: `dtypes == "object"` gives a True/False value per column. Indexing the dtypes by that mask and taking `.index` returns the names of the text columns, which I can then use to select just those.

In [4]:
# select the object (text) columns using a boolean mask on dtypes
a = titanic_train.dtypes[titanic_train.dtypes == "object"].index
a

Index(['Name', 'Sex', 'Cabin', 'Embarked'], dtype='str')

`describe()` summarises columns. On text columns it reports count, number of unique values, the most common value (top), and how often it appears (freq). On numeric columns it reports mean, standard deviation, min, max and the quartiles instead.

In [5]:
# describe on text columns - count, unique, top, freq
titanic_train[a].describe()

,Name,Sex,Cabin,Embarked
count,150,150,150,150
unique,150,2,8,3
top,Sylvia Ellis,female,F2,C
freq,1,75,22,52


In [6]:
# describe on numeric columns - mean, std, min, max, quartiles
titanic_train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,150.000000,150.000000,150.000000,150.00000,150.000000,150.000000,150.000000
mean,75.500000,0.520000,2.026667,29.88200,2.593333,2.453333,57.800733
std,43.445368,0.501274,0.843026,17.85034,1.742083,1.815617,25.761402
min,1.000000,0.000000,1.000000,1.00000,0.000000,0.000000,10.300000
25%,38.250000,0.000000,1.000000,13.62500,1.000000,1.000000,37.307500
50%,75.500000,1.000000,2.000000,29.40000,2.000000,2.000000,54.895000
75%,112.750000,1.000000,3.000000,44.80000,4.000000,4.000000,83.112500
max,150.000000,1.000000,3.000000,60.00000,5.000000,5.000000,99.660000


### Selecting Columns and Rows
One column with `df['col']` returns a Series. Several columns need a list of names, so double brackets: `df[['Name','Sex','Age']]`. `unique()` lists the distinct values in a column, which is useful for spotting categories and stray values (nan shows up if there are missing entries).

In [7]:
# many columns - pass a list of names (double brackets)
titanic_train[['Name', 'Sex', 'Age']].head()

,Name,Sex,Age
0,Sylvia Ellis,female,21.8
1,Kim Skinner,male,36.8
2,Joseph Marsh,female,50.8
3,Stephanie Cox,male,41.7
4,Tina Cox,female,3.1


In [8]:
# unique() - the distinct values in a column
titanic_train['Cabin'].unique()

array(['F2', 'G6', 'nan', 'D33', 'C123', 'A10', 'B20', 'E46'],
      dtype=object)

### np.where, Finding Rows
`np.where(condition)` returns the positions where a condition is True. Feeding those positions into `iloc` pulls out the matching rows. Below I build a Family column (siblings/spouses plus parents/children) and find the rows with the largest family.

In [9]:
# isnull().sum() - count missing values in a column
titanic_train['Age'].isnull().sum()

np.int64(0)

In [10]:
# np.where returns positions, feed into iloc to pull the matching rows
titanic_train['Family'] = titanic_train['SibSp'] + titanic_train['Parch']
biggest = np.where(titanic_train['Family'] == max(titanic_train['Family']))
titanic_train.iloc[biggest]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,Family
28,29,1,1,Phillip Johnson,female,15.1,5,5,60.11,B20,S,10
58,59,1,3,Kim Marsh,female,2.0,5,5,37.08,nan,S,10
59,60,1,3,Phillip Skinner,female,23.2,5,5,61.85,B20,C,10
60,61,0,3,Jose Wilson,male,23.6,5,5,10.30,A10,C,10
61,62,0,2,Tina Mercado,male,13.6,5,5,61.11,F2,S,10
76,77,1,3,Theresa Hernandez,male,24.2,5,5,47.64,E46,C,10
93,94,0,2,Jimmy Murray,male,52.7,5,5,32.11,E46,S,10
102,103,1,2,Joseph Schmidt,female,29.7,5,5,29.76,A10,Q,10
142,143,0,1,Cody Wilson,male,43.6,5,5,83.23,nan,S,10
143,144,1,2,Tammy Adams,male,43.9,5,5,50.50,C123,S,10


### Series Basics
A Series is a single column with an index. I can build one from a list with a custom index, or from a dict (keys become the index). The important behaviour is alignment: when I combine two Series, pandas matches them on the index. Where a label exists in one but not the other, the result is NaN. This automatic alignment is a big reason pandas is safer than plain lists.

In [11]:
# Series from a list with a custom index
pd.Series([10, 20, 30], index=['a', 'b', 'c'])

a    10
b    20
c    30
dtype: int64

In [12]:
# alignment - adding two Series matches on index, NaN where there is no match
ser1 = pd.Series([1, 2, 3, 4], ['CA', 'OR', 'CO', 'CA'])
ser2 = pd.Series([1, 2, 5, 4], ['CA', 'NV', 'AZ', 'OR'])
ser1 + ser2

AZ    NaN
CA    2.0
CA    5.0
CO    NaN
NV    NaN
OR    6.0
dtype: float64

### DataFrame, loc vs iloc
Two ways to select from a DataFrame, and mixing them up is a common bug:
- `loc[row_label, col_label]` selects by label (the names)
- `iloc[row_pos, col_pos]` selects by position (integer location, 0 based)

I can also create a new column from existing ones, and remove rows or columns with `drop` (axis=0 for a row, axis=1 for a column).

In [13]:
# small DataFrame of random numbers
np.random.seed(101)
df = pd.DataFrame(np.random.randn(5, 4), index=['A','B','C','D','E'], columns=['W','X','Y','Z'])
df

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [14]:
# loc is by label, iloc is by position
print(df.loc['C', 'W'])
print(df.iloc[1, 2])

-2.018168244037392
-0.8480769834036315


In [15]:
# new column built from existing ones
df['New'] = df['X'] + df['Z']
df

,W,X,Y,Z,New
A,2.706850,0.628133,0.907969,0.503826,1.131958
B,0.651118,-0.319318,-0.848077,0.605965,0.286647
C,-2.018168,0.740122,0.528813,-0.589001,0.151122
D,0.188695,-0.758872,-0.933237,0.955057,0.196184
E,0.190794,1.978757,2.605967,0.683509,2.662266


In [16]:
# drop - axis=1 drops a column (axis=0 would drop a row)
df.drop('New', axis=1)

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


### Conditional Selection
The everyday way to filter: build a True/False mask from a condition on a column, then index the DataFrame with it to keep only matching rows. Combine conditions with `&` (and) and `|` (or), and wrap each condition in its own parentheses, otherwise Python's operator precedence gives the wrong result.

In [17]:
# boolean mask filters rows, combine conditions with & and | in parentheses
data = pd.DataFrame(np.matrix('22,66,140;42,70,148;30,62,125;35,68,160;25,62,152'),
                    index=['A','B','C','D','E'], columns=['Age','Height','Weight'])
data[(data['Height'] > 65) & (data['Weight'] > 145)]

,Age,Height,Weight
B,42,70,148
D,35,68,160


### Missing Data
Real data has gaps (NaN). Two main tools:
- `dropna()` removes missing values. `axis=0` drops rows that contain NaN, `axis=1` drops columns. `thresh=n` keeps a row only if it has at least n non NaN values.
- `fillna(value=...)` replaces NaN with something: a constant, or a computed value like the column mean (a common simple imputation).

In [18]:
# a frame with NaNs
mdf = pd.DataFrame({'A':[1,2,np.nan], 'B':[5,np.nan,np.nan], 'C':[1,2,3]})
mdf

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2
2,NaN,NaN,3


In [19]:
# dropna with thresh - keep only rows that have at least 2 non-NaN values
mdf.dropna(thresh=2)

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2


In [20]:
# fillna - replace NaN, here with the mean of column A
mdf.fillna(value=mdf['A'].mean())

,A,B,C
0,1.0,5.0,1
1,2.0,1.5,2
2,1.5,1.5,3


### groupby
groupby follows the split, apply, combine pattern: split the rows into groups by some column, apply an aggregate (mean, sum, count, describe) to each group, then combine the results into one table. Below, average sales per company.

In [21]:
# groupby - split into groups, then aggregate each group
sales = pd.DataFrame({'Company':['GOOG','GOOG','MSFT','MSFT','FB','FB'],
                      'Person':['Sam','Charlie','Amy','Vanessa','Carl','Sarah'],
                      'Sales':[200,120,340,124,243,350]})
sales.groupby('Company').mean(numeric_only=True)

,Sales
Company,
FB,296.5
GOOG,160.0
MSFT,232.0


### concat, merge, join
Three ways to combine DataFrames:
- `concat` glues frames together. `axis=0` stacks them (more rows), `axis=1` puts them side by side (more columns).
- `merge` joins two frames on a shared key column, exactly like a SQL join. `how=` controls which keys to keep: inner (only matches), left, right, or outer (all).
- `join` is like merge but it joins on the index instead of a column.

In [22]:
# concat - stack rows
df1 = pd.DataFrame({'A':['A0','A1'], 'B':['B0','B1']})
df2 = pd.DataFrame({'A':['A2','A3'], 'B':['B2','B3']})
pd.concat([df1, df2])

,A,B
0,A0,B0
1,A1,B1
0,A2,B2
1,A3,B3


In [23]:
# merge - join two frames on a key column
left = pd.DataFrame({'key':['K0','K1','K2'], 'A':['A0','A1','A2']})
right = pd.DataFrame({'key':['K0','K1','K2'], 'B':['B0','B1','B2']})
pd.merge(left, right, how='inner', on='key')

,key,A,B
0,K0,A0,B0
1,K1,A1,B1
2,K2,A2,B2


In [24]:
# join - like merge but on the index. how='outer' keeps all index labels
l = pd.DataFrame({'A':['A0','A1']}, index=['K0','K1'])
r = pd.DataFrame({'B':['B0','B2']}, index=['K0','K2'])
l.join(r, how='outer')

,A,B
K0,A0,B0
K1,A1,NaN
K2,NaN,B2


### apply and sort
`apply` runs a function on every value of a column: a named function, a one line `lambda`, or a built in like `len`. After transforming, the usual aggregates (sum, mean, std) work. `sort_values(by=...)` orders rows by a column, `ascending=False` for high to low.

In [25]:
# apply - run a function over a column (lambda, or built in like len)
adf = pd.DataFrame({'col1':[1,2,3,4], 'col2':[444,555,666,444], 'col3':['aaa','bb','c','dddd']})
adf['log_col2'] = adf['col2'].apply(lambda x: np.log(x))
adf['len_col3'] = adf['col3'].apply(len)
adf

,col1,col2,col3,log_col2,len_col3
0,1,444,aaa,6.095825,3
1,2,555,bb,6.318968,2
2,3,666,c,6.501290,1
3,4,444,dddd,6.095825,4


In [26]:
# sort_values - order rows by a column
adf.sort_values(by='col2', ascending=False)

,col1,col2,col3,log_col2,len_col3
2,3,666,c,6.501290,1
1,2,555,bb,6.318968,2
0,1,444,aaa,6.095825,3
3,4,444,dddd,6.095825,4


### Recap
- Series is 1D, DataFrame is 2D, both align on the index
- select with `df['col']`, `df[['a','b']]`, `loc` (label) and `iloc` (position)
- filter rows with boolean masks, combine with `&` and `|` in parentheses
- clean with `dropna` and `fillna`
- combine with `concat` (stack/widen), `merge` (on a key), `join` (on the index)
- transform with `groupby` aggregates and `apply`